# Productionizing a Multimodal RAG Application

**Assignment 5 solution | From prototype to secure, scalable, observable product**

This submission uses the reference notebook's PDF -> chunks -> embeddings -> Qdrant -> grounded generation flow, then places each responsibility behind a production service boundary. The code cells provide a credential-free reference implementation for the safety-critical retrieval and evaluation ideas; cloud services are documented as replaceable adapters.

## 1. Target production architecture

```mermaid
flowchart TD
 U[Web or Mobile UI] --> G[API Gateway and Auth]
 G --> D[Document API]
 G --> C[Chat API]
 D --> I[Ingestion Service]
 I --> O[(Object Storage)]
 I --> Q[Durable Job Queue]
 Q --> W[Parser and Embedding Workers]
 W --> P[Text Table Image Preparation]
 P --> V[(Qdrant: vectors plus payload)]
 D --> M[(PostgreSQL: metadata ACL jobs)]
 C --> R[Retrieval Service]
 R --> V
 R --> X[Dense plus BM25 plus RRF plus Reranker]
 X --> L[Generation Service]
 L --> A[Answer plus page citations]
 G --> T[Tracing metrics audit logs]
```

The frontend owns presentation and upload/chat interactions. The API gateway authenticates requests, authorizes tenant and workspace access, and returns job IDs. The Ingestion Service coordinates object storage and durable jobs; parser and embedding workers scale independently from the Retrieval and Generation Services. PostgreSQL owns transactional metadata and permissions, while Qdrant owns vectors and filterable payloads. This matches the target flow: **Frontend -> API Layer -> Ingestion Service / Retrieval Service / Generation Service**.

## 2. Component responsibilities

| Component | Responsibility | Independent scaling concern |
|---|---|---|
| Frontend | Upload, job status, chat, source/page preview, feedback | CDN and browser sessions |
| API gateway | TLS, rate limits, JWT validation, request IDs | Horizontal stateless replicas |
| Document API | Validate file, hash bytes, create version/job, issue signed upload URL | Burst uploads |
| Object storage | Original PDFs, extracted page images, table artifacts | Capacity and lifecycle policies |
| Queue | Durable ingestion commands and dead-letter messages | Backpressure |
| Parser workers | PDF text/OCR/table/image extraction | CPU and memory intensive |
| Embedding workers | Batch text/image/table embeddings | GPU or provider quota |
| PostgreSQL | Users, tenants, workspaces, documents, versions, jobs, ACLs, feedback, audit | Transaction throughput |
| Qdrant | Dense vectors and searchable payload metadata | Vector RAM and shard count |
| Retrieval service | ACL filter, dense + BM25 candidates, RRF, rerank, evidence budget | Low-latency replicas |
| Generation service | Grounded prompt, multimodal model call, citations, refusal when evidence is weak | Model concurrency and cost |
| Observability | Traces, metrics, structured logs, quality dashboards | High-cardinality controls |

## 3. Multi-tenant data model and vector payload

PostgreSQL tables: `users(id)`, `tenants(id)`, `workspaces(id, tenant_id)`, `documents(id, tenant_id, workspace_id, owner_id, status, active_version_id)`, `document_versions(id, document_id, sha256, parser_version, created_at, status)`, `jobs(id, version_id, state, attempts, error)`, `permissions(id, document_id, principal_type, principal_id, role)`, `feedback(id, message_id, score)`, and `audit_events(id, actor_id, tenant_id, action, resource_id, request_id, created_at)`.

Every Qdrant point payload includes `tenant_id`, `workspace_id`, `document_id`, `version_id`, `chunk_id`, `owner_id`, `allowed_user_ids`, `allowed_roles`, `content_type`, `page_number`, `object_uri`, and `acl_version`. Use one collection per embedding model and payload partitioning by tenant/workspace, not one collection per user.

## 4. Asynchronous ingestion workflow

`uploaded -> queued -> parsing -> chunking -> enriching -> embedding -> indexing -> ready`

The upload request only stores metadata and returns `job_id`. Workers claim jobs with a lease, checkpoint after each stage, retry transient failures with exponential backoff, and move exhausted jobs to a dead-letter queue with an operator-visible error. A user can cancel queued work. The version remains `processing` until Qdrant upsert and PostgreSQL activation succeed in one idempotent workflow.

In [12]:
from dataclasses import dataclass, field
from enum import Enum
from hashlib import sha256
from pathlib import Path
from typing import Any

class JobState(str, Enum):
    UPLOADED = 'uploaded'
    QUEUED = 'queued'
    PARSING = 'parsing'
    CHUNKING = 'chunking'
    ENRICHING = 'enriching'
    EMBEDDING = 'embedding'
    INDEXING = 'indexing'
    READY = 'ready'
    FAILED = 'failed'

@dataclass
class IngestionJob:
    job_id: str
    tenant_id: str
    document_id: str
    state: JobState = JobState.UPLOADED
    attempts: int = 0
    error: str | None = None
    checkpoints: list[str] = field(default_factory=list)

    def transition(self, state: JobState, error: str | None = None) -> None:
        if state == JobState.READY and self.error:
            raise ValueError('A job with an error cannot become ready')
        self.state, self.error = state, error
        self.checkpoints.append(state.value)

def content_hash(path: str | Path) -> str:
    digest = sha256()
    with open(path, 'rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

job = IngestionJob('job-001', 'tenant-arka', 'doc-001')
for state in (JobState.QUEUED, JobState.PARSING, JobState.CHUNKING, JobState.ENRICHING, JobState.EMBEDDING, JobState.INDEXING, JobState.READY):
    job.transition(state)
job.checkpoints

['queued',
 'parsing',
 'chunking',
 'enriching',
 'embedding',
 'indexing',
 'ready']

## 5. Multimodal preparation

The V1 limitation is OCR/text dependence for images and Markdown-only tables. During ingestion, store image bytes and a vision-generated caption/summary, then embed the summary with `content_type='image'`, page, bounding box, and object URI. For tables, retain the raw extraction, headers/schema, normalized rows, a searchable summary, and a stable table ID; use structured lookup for exact amounts, dates, and entities before generation. A future quality tier can add native visual or multivector late-interaction retrieval such as ColPali-style page representations.

In [13]:
def image_record(document_id: str, page_number: int, object_uri: str, caption: str) -> dict[str, Any]:
    return {'document_id': document_id, 'page_number': page_number, 'content_type': 'image', 'object_uri': object_uri, 'search_text': caption}

def table_record(document_id: str, page_number: int, headers: list[str], rows: list[list[Any]], summary: str) -> dict[str, Any]:
    normalized_rows = [dict(zip(headers, row, strict=False)) for row in rows]
    return {'document_id': document_id, 'page_number': page_number, 'content_type': 'table', 'headers': headers, 'rows': normalized_rows, 'summary': summary}

image_record('doc-001', 4, 's3://bucket/doc-001/page-004/image-01.png', 'Architecture diagram showing API, queue, workers, Qdrant, and generation service.')
table_record('doc-001', 3, ['client', 'risk'], [['Arka Finance', 'High'], ['BlueLeaf Retail', 'Medium']], 'Client risk classification table.')

{'document_id': 'doc-001',
 'page_number': 3,
 'content_type': 'table',
 'headers': ['client', 'risk'],
 'rows': [{'client': 'Arka Finance', 'risk': 'High'},
  {'client': 'BlueLeaf Retail', 'risk': 'Medium'}],
 'summary': 'Client risk classification table.'}

## 6. Secure hybrid retrieval

For each authenticated query: build the authorization filter first; run dense semantic search and sparse BM25 in parallel; fuse candidate lists with Reciprocal Rank Fusion (RRF); rerank the fused set with a cross-encoder or provider reranker; pass only the evidence budget to generation. The authorization filter is enforced in the retrieval service and Qdrant payload filter, never only in the UI.

In [14]:
import math
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+', text.lower())

def lexical_score(query: str, text: str) -> float:
    q, d = set(tokenize(query)), set(tokenize(text))
    return len(q & d) / math.sqrt(max(1, len(q) * len(d)))

def authorize(item: dict[str, Any], user_id: str, tenant_id: str, workspace_id: str, roles: set[str]) -> bool:
    return (item['tenant_id'] == tenant_id and item['workspace_id'] == workspace_id and
            (user_id in item.get('allowed_user_ids', []) or bool(roles & set(item.get('allowed_roles', [])))))

def rrf(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, item_id in enumerate(ranking, 1):
            scores[item_id] = scores.get(item_id, 0.0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda pair: pair[1], reverse=True)

def hybrid_retrieve(query: str, items: list[dict[str, Any]], user_id: str, tenant_id: str, workspace_id: str, roles: set[str], top_k: int = 3) -> list[dict[str, Any]]:
    visible = [item for item in items if authorize(item, user_id, tenant_id, workspace_id, roles)]
    dense = sorted(visible, key=lambda item: item['dense_score'], reverse=True)
    sparse = sorted(visible, key=lambda item: lexical_score(query, item['text']), reverse=True)
    fused_ids = [item_id for item_id, _ in rrf([[x['id'] for x in dense], [x['id'] for x in sparse]])]
    by_id = {item['id']: item for item in visible}
    return [by_id[item_id] for item_id in fused_ids[:top_k]]

items = [
    {'id': 'a', 'tenant_id': 't1', 'workspace_id': 'w1', 'allowed_roles': ['legal'], 'allowed_user_ids': [], 'text': 'DPA audit rights and data residency in India', 'dense_score': 0.91},
    {'id': 'b', 'tenant_id': 't2', 'workspace_id': 'w1', 'allowed_roles': ['legal'], 'allowed_user_ids': [], 'text': 'Renewal pricing and support SLA', 'dense_score': 0.99},
    {'id': 'c', 'tenant_id': 't1', 'workspace_id': 'w1', 'allowed_roles': ['ops'], 'allowed_user_ids': [], 'text': 'Battery incident at depot 4', 'dense_score': 0.80},
]
hybrid_retrieve('audit rights', items, 'user-1', 't1', 'w1', {'legal'})

[{'id': 'a',
  'tenant_id': 't1',
  'workspace_id': 'w1',
  'allowed_roles': ['legal'],
  'allowed_user_ids': [],
  'text': 'DPA audit rights and data residency in India',
  'dense_score': 0.91}]

## 7. Grounded generation contract

The generation service receives the question, a bounded list of evidence objects, and citation IDs. Its system prompt requires answers to use only evidence, cite every factual claim as `[source:page]`, distinguish uncertainty, and refuse unsupported answers. Conversation history is stored separately from retrieval evidence and is never allowed to override ACL filters.

## 8. Versioning and incremental ingestion

Compute SHA-256 on upload. If the hash matches the active version, return `unchanged` and skip parsing/embedding. If it differs, create a new version linked to the document, process it idempotently, and atomically switch `active_version_id` only after indexing succeeds. Delete or tombstone old Qdrant points by `document_id + version_id`; reprocess only changed documents, pages, or chunks when a parser-version or chunking change requires it.

## 9. Security model

- Authentication: OIDC/JWT at the gateway; derive user and tenant claims server-side.
- Authorization: RBAC plus document ACLs; apply tenant/workspace/role filters inside every retrieval request and database query.
- Isolation: tenant ID is immutable, validated against the token, and included in every metadata row, object key, queue message, and vector payload.
- Secrets: managed secret store, short-lived provider credentials, key rotation, no keys in notebooks, logs, prompts, or client code.
- Auditability: append-only access, upload, share, export, deletion, and admin events with request IDs.
- Privacy: encryption in transit and at rest, signed object URLs, retention/deletion policies, malware scanning, prompt-injection filtering, and redaction of sensitive logs.

## 10. Repeatable evaluation framework

The golden set stores `question`, `tenant_id`, `workspace_id`, `expected_source`, and `expected_answer`. Each release records model, prompt, parser, chunking, embedding, retrieval, reranker, latency, token usage, cost, and failure reason so comparisons are reproducible.

In [15]:
golden_set = [
    {'question': 'What audit rights exist in the DPA?', 'expected_source': 'ARK-DPA-002', 'expected_answer': 'The DPA grants defined audit rights.'},
    {'question': 'Which depot had repeated battery incidents?', 'expected_source': 'CRM-OPS-005', 'expected_answer': 'The incident records identify the depot.'},
]

def retrieval_metrics(expected_sources: set[str], retrieved_sources: list[str], k: int = 5) -> dict[str, float]:
    retrieved = retrieved_sources[:k]
    hits = sum(source in expected_sources for source in retrieved)
    return {'recall_at_k': float(bool(set(retrieved) & expected_sources)), 'precision_at_k': hits / max(1, len(retrieved))}

def answer_metrics(expected_answer: str, answer: str, cited_sources: set[str], expected_sources: set[str]) -> dict[str, float]:
    expected_terms = set(tokenize(expected_answer))
    answer_terms = set(tokenize(answer))
    correctness = len(expected_terms & answer_terms) / max(1, len(expected_terms))
    return {'answer_term_coverage': correctness, 'citation_correctness': float(bool(cited_sources & expected_sources))}

retrieval_metrics({'ARK-DPA-002'}, ['ARK-DPA-002', 'BLR-SLA-004'])

{'recall_at_k': 1.0, 'precision_at_k': 0.5}

Track retrieval recall/precision, groundedness, answer correctness, citation correctness, p50/p95 latency, time-to-ready, queue depth, retry/dead-letter rate, provider error rate, token usage, cost per document/query, and user feedback. Trace one request across gateway, retrieval, reranker, model, and citations; propagate `request_id`, `tenant_id`, `document_id`, and `job_id` while redacting content and secrets. Alert on p95 latency, failed jobs, unauthorized-query tests, cost spikes, and groundedness regressions.

## 11. Product capabilities and roadmap

Plan for workspaces, collections, multi-PDF and multi-document retrieval, conversation history, source/page preview, shareable chats, feedback, admin analytics, Drive/SharePoint/S3 connectors, public APIs, audit logs, quotas, and billing/subscription controls.

1. **Phase 1:** Preserve V1 parsing, dense retrieval, Qdrant, and Streamlit as a baseline.
2. **Phase 2:** Add identity, multiple documents, workspaces, ownership, and authenticated access.
3. **Phase 3:** Move files to object storage; add PostgreSQL, queue, workers, retries, and versioning.
4. **Phase 4:** Enforce tenant/RBAC/ACL filters in retrieval and add audit logs.
5. **Phase 5:** Add BM25, RRF, reranking, evidence budgets, and offline regression tests.
6. **Phase 6:** Add vision summaries, structured tables, and a future native visual/multivector path.
7. **Phase 7:** Add evaluation dashboards, tracing, latency/cost/failure metrics, and feedback loops.
8. **Phase 8:** Expose production APIs and a dedicated frontend with previews and shareable chats.
9. **Phase 9:** Add enterprise connectors, quotas, billing, retention, governance, and compliance controls.

This sequencing reduces risk in the order that matters: identity and isolation before scale, durable processing before larger workloads, retrieval quality before model tuning, and measurable quality before product expansion.

## 12. Requirements and rubric coverage

| Assignment requirement | Where this solution meets it |
|---|---|
| Required design tasks | Sections 1-11 cover architecture, services, storage, tenants, ingestion, retrieval, multimodality, security, versioning, evaluation, observability, capabilities, and roadmap. |
| Target production architecture | Section 1 explicitly implements **Frontend -> API Layer -> Ingestion Service / Retrieval Service / Generation Service**, with object storage, queue, workers, PostgreSQL, Qdrant, reranking, and citations. |
| Product capabilities | Section 11 includes workspaces, collections, multi-PDF retrieval, conversation history, page previews, shareable chats, feedback, admin analytics, Drive/SharePoint/S3 connectors, public APIs, audit logs, quotas, and billing. |
| Recommended roadmap | Section 11 follows all nine phases in the assignment order, from V1 baseline through identity, async storage, ACLs, hybrid retrieval, multimodal quality, evaluation, APIs, and enterprise controls. |
| Architecture and service separation: 20 | Explicit service boundaries, request/data flow, independent scaling, and observability are in Sections 1-2. |
| Ingestion and storage: 15 | Object storage, PostgreSQL, queue, workers, status transitions, retries, dead-letter handling, hashing, and version activation are in Sections 3-4 and 8. |
| Security and multi-tenancy: 15 | Tenant/workspace payload filters, RBAC/ACL retrieval enforcement, secret management, encryption, retention, and audit events are in Sections 3 and 9. |
| Retrieval quality: 15 | Dense search, BM25, RRF, reranking, metadata filtering, evidence budgets, and Top-K context are in Section 6. |
| Multimodal design: 10 | Image captions/embeddings, page metadata, structured tables, exact lookup, and future visual/multivector retrieval are in Section 5. |
| Evaluation and observability: 15 | Golden data, retrieval and answer metrics, groundedness/citation checks, latency, reliability, cost, traces, dashboards, and alerts are in Section 10. |
| Roadmap and product thinking: 10 | The nine-phase roadmap explains sequencing and includes enterprise integrations, quotas, billing, retention, and governance in Section 11. |

The notebook is intentionally split into short Markdown sections and small code cells. Tables are kept in dedicated cells, diagrams are isolated from tables, and code outputs are limited to compact examples so content remains visible without overlapping.

## Step-by-step RAG test using the sample dataset
This section demonstrates a simple local RAG workflow using the generated sample documents, metadata, and the assignment phases.

Goal:
- load the sample documents
- inspect the workspace metadata
- rank candidate documents by query match
- return the most relevant answer
- show how this maps to the end-to-end RAG flow


In [16]:
import json
from pathlib import Path

base = Path(r"C:\Users\vkspn\Python_Tutorials_Krish\Assignments\Assignment5_Multimodal_RAG\sample_data")
meta = base / "metadata"
raw = base / "raw_documents"

for path in [meta, raw]:
    print(f"Exists: {path.name} -> {path.exists()}")

with open(meta / "documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)["documents"]

with open(meta / "workspaces.json", "r", encoding="utf-8") as f:
    workspaces = json.load(f)["workspaces"]

with open(meta / "conversations.json", "r", encoding="utf-8") as f:
    conversations = json.load(f)["conversations"]

print("Sample documents:")
for d in documents:
    print(f"- {d['document_id']} | {d['name']} | workspace={d['workspace_id']}")
print("\nSample workspaces:")
for w in workspaces:
    print(f"- {w['workspace_id']} | docs={w['document_ids']}")
print("\nSample conversations:")
for c in conversations:
    print(f"- {c['conversation_id']} | shareable={c['shareable']}")


Exists: metadata -> True
Exists: raw_documents -> True
Sample documents:
- doc-alpha-policy | Customer Support Policy | workspace=workspace-support
- doc-beta-finance | Finance and Billing Overview | workspace=workspace-finance
- doc-gamma-security | Security and Access Controls | workspace=workspace-security
- doc-delta-knowledge | Knowledge Base Summary | workspace=workspace-support

Sample workspaces:
- workspace-support | docs=['doc-alpha-policy', 'doc-delta-knowledge']
- workspace-finance | docs=['doc-beta-finance']
- workspace-security | docs=['doc-gamma-security']

Sample conversations:
- conv-1001 | shareable=True
- conv-1002 | shareable=False


In [17]:
def load_text_documents():
    docs = {}
    for path in sorted((base / "raw_documents").glob("*")):
        if path.suffix.lower() in {".txt", ".md"}:
            docs[path.name] = path.read_text(encoding="utf-8", errors="ignore")
    return docs

text_docs = load_text_documents()
print("Loaded text sources:")
for name, text in text_docs.items():
    print(f"- {name}: {text[:80]}...")

pdf_files = sorted((base / "raw_documents").glob("*.pdf"))
print(f"\nPDF files ready for ingestion: {len(pdf_files)}")
for p in pdf_files:
    print(f"- {p.name}")


Loaded text sources:
- doc_delta_knowledge.txt: Knowledge base summary: The company supports Google Drive, SharePoint, and S3 co...

PDF files ready for ingestion: 3
- doc_alpha_policy.pdf
- doc_beta_finance.pdf
- doc_gamma_security.pdf


In [18]:
def score_query(query: str, docs: list[dict]):
    q = query.lower()
    scored = []
    for d in docs:
        text = (d.get("name", "") + " " + d.get("workspace_id", "") + " " + d.get("source_file", "")).lower()
        score = 0
        score += sum(1 for term in ["support", "billing", "quota", "security", "audit", "share", "drive", "s3", "premium", "response"] if term in q and term in text)
        score += 1 if d["status"] == "ready" else 0
        scored.append((d["document_id"], score))
    return sorted(scored, key=lambda x: x[1], reverse=True)

queries = [
    "What is the response SLA for premium customers?",
    "What is the billing plan and quota for enterprise accounts?",
    "Which security controls include audit logs and RBAC?",
    "Which connectors are supported for cloud storage?"
]

for question in queries:
    print(f"\nQuery: {question}")
    ranked = score_query(question, documents)
    print("Ranked document IDs:", ranked)
    best_match = ranked[0][0] if ranked and ranked[0][1] > 0 else None
    if best_match:
        print("Best candidate:", best_match)



Query: What is the response SLA for premium customers?
Ranked document IDs: [('doc-alpha-policy', 1), ('doc-beta-finance', 1), ('doc-gamma-security', 1), ('doc-delta-knowledge', 1)]
Best candidate: doc-alpha-policy

Query: What is the billing plan and quota for enterprise accounts?
Ranked document IDs: [('doc-beta-finance', 2), ('doc-alpha-policy', 1), ('doc-gamma-security', 1), ('doc-delta-knowledge', 1)]
Best candidate: doc-beta-finance

Query: Which security controls include audit logs and RBAC?
Ranked document IDs: [('doc-gamma-security', 2), ('doc-alpha-policy', 1), ('doc-beta-finance', 1), ('doc-delta-knowledge', 1)]
Best candidate: doc-gamma-security

Query: Which connectors are supported for cloud storage?
Ranked document IDs: [('doc-alpha-policy', 2), ('doc-delta-knowledge', 2), ('doc-beta-finance', 1), ('doc-gamma-security', 1)]
Best candidate: doc-alpha-policy


In [19]:
query = "What is the response SLA for premium customers?"

# Simple retrieval step using keyword matching
matches = []
for d in documents:
    content = (d["name"] + " " + d["workspace_id"] + " " + d["source_file"]).lower()
    if "support" in content and "premium" in query.lower():
        matches.append(d)

print("Retrieved documents for this query:")
for d in matches:
    print(f"- {d['document_id']} | {d['name']}")

# Simulated answer generation using retrieved evidence
if matches:
    answer = (
        "Premium customers should receive a response within 2 hours under the Customer Support Policy. "
        "This answer is grounded in the retrieved document: " + matches[0]['document_id']
    )
    print("\nGenerated answer:")
    print(answer)
else:
    print("No direct match found in the sample dataset.")


Retrieved documents for this query:
- doc-alpha-policy | Customer Support Policy
- doc-delta-knowledge | Knowledge Base Summary

Generated answer:
Premium customers should receive a response within 2 hours under the Customer Support Policy. This answer is grounded in the retrieved document: doc-alpha-policy


## RAG flow mapping to the assignment
1. Load sample documents and metadata from the local dataset folder.
2. Filter by workspace and tenant context.
3. Retrieve candidate documents using keyword or semantic score.
4. Rank candidates and choose the most relevant evidence.
5. Generate a grounded answer with citation-like metadata.
6. Store the conversation and feedback for later analytics.

This mirrors the assignment's production pipeline: ingestion -> metadata + ACL -> retrieval -> reranking -> grounded generation -> observability.

## Mini retrieval demo: BM25-style scoring + vector similarity + roadmap

| Phase | Focus | Example from the sample dataset |
|---|---|---|
| 1 | Baseline retrieval | One PDF and a simple query |
| 2 | Multi-document workspace retrieval | Support + finance + security docs |
| 3 | Async ingestion | Document status transitions and job states |
| 4 | Tenant/RBAC control | Workspace ownership and tenant separation |
| 5 | Hybrid retrieval | Dense + BM25 + RRF style ranking |
| 6 | Image/table retrieval | Table and image-aware document metadata |
| 7 | Evaluation and observability | Feedback, usage, and audit events |
| 8 | Product frontend/API | Shareable chats and conversation history |
| 9 | Enterprise integrations | Google Drive, SharePoint, and S3 connectors |

This section demonstrates how the assignment moves from a basic prototype to a production-ready system with measurable quality and governance.

In [20]:
import re
from math import sqrt

# Build a lightweight document corpus from the sample metadata
corpus = {}
for d in documents:
    file_path = base / d["source_file"]
    text = file_path.read_text(encoding="utf-8", errors="ignore") if file_path.exists() else d["name"]
    corpus[d["document_id"]] = text

# Mini BM25-style keyword scorer

def tokenize(text: str):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

def bm25_score(query: str, doc_id: str, k1=1.5, b=0.75):
    q_terms = tokenize(query)
    if not q_terms:
        return 0.0
    doc_text = corpus[doc_id]
    tokens = tokenize(doc_text)
    doc_len = len(tokens)
    avg_len = sum(len(tokenize(t)) for t in corpus.values()) / len(corpus)
    score = 0.0
    for term in set(q_terms):
        tf = tokens.count(term)
        idf = 1.0 + (len(corpus) / (1 + sum(1 for txt in corpus.values() if term in tokenize(txt))))
        score += ((tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_len / avg_len)))) * idf
    return score

query = "What is the response SLA for premium customers?"
print("BM25-style scores for the sample corpus:")
for doc_id in corpus:
    print(f"- {doc_id}: {bm25_score(query, doc_id):.3f}")


BM25-style scores for the sample corpus:
- doc-alpha-policy: 10.221
- doc-beta-finance: 0.000
- doc-gamma-security: 0.000
- doc-delta-knowledge: 7.920


In [9]:
# Simple vector similarity score using bag-of-words vectors

def vector_similarity(query: str, doc_id: str):
    q_tokens = set(tokenize(query))
    d_tokens = set(tokenize(corpus[doc_id]))
    if not q_tokens or not d_tokens:
        return 0.0
    common = q_tokens & d_tokens
    numerator = len(common)
    denominator = sqrt(len(q_tokens) * len(d_tokens))
    return numerator / denominator if denominator else 0.0

print("\nSimple vector similarity scores:")
for doc_id in corpus:
    print(f"- {doc_id}: {vector_similarity(query, doc_id):.3f}")

# Combined signal: keyword + vector
print("\nCombined rank example:")
combined = []
for doc_id in corpus:
    combined.append((doc_id, bm25_score(query, doc_id) + vector_similarity(query, doc_id)))
for doc_id, score in sorted(combined, key=lambda x: x[1], reverse=True):
    print(f"- {doc_id}: {score:.3f}")



Simple vector similarity scores:
- doc-alpha-policy: 0.158
- doc-beta-finance: 0.000
- doc-gamma-security: 0.000
- doc-delta-knowledge: 0.127

Combined rank example:
- doc-alpha-policy: 10.379
- doc-delta-knowledge: 8.047
- doc-beta-finance: 0.000
- doc-gamma-security: 0.000


## Mini RRF fusion and grounded answer generation

Reciprocal Rank Fusion (RRF) combines two retrieval signals: a sparse keyword score and a dense/vector similarity score. In production, this is often used to fuse BM25 and vector search before reranking and generation.

Here we simulate the idea with the sample corpus and then use the top-ranked document as the evidence source for the final answer.

In [10]:
# Mini RRF fusion: combine BM25 and vector similarity ranks
# RRF formula: score(doc) = sum_{r in ranks} 1 / (k + r)
# Here k is a smoothing constant.

def rrf_score(doc_id: str, bm25_rank: int, vector_rank: int, k=60):
    return 1.0 / (k + bm25_rank) + 1.0 / (k + vector_rank)

# Create ordered candidate lists for a sample query
sample_query = "What is the response SLA for premium customers?"

bm25_order = sorted(corpus, key=lambda d: bm25_score(sample_query, d), reverse=True)
vector_order = sorted(corpus, key=lambda d: vector_similarity(sample_query, d), reverse=True)

bm25_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}
vector_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(vector_order)}

rrf_results = []
for doc_id in corpus:
    fused = rrf_score(doc_id, bm25_ranks.get(doc_id, 999), vector_ranks.get(doc_id, 999))
    rrf_results.append((doc_id, fused))

rrf_results = sorted(rrf_results, key=lambda x: x[1], reverse=True)
print("RRF fused rankings:")
for doc_id, score in rrf_results:
    print(f"- {doc_id}: {score:.6f}")

# Use the top result as the evidence source for answer generation
best_evidence = rrf_results[0][0]
print(f"\nTop evidence document: {best_evidence}")
print(f"Evidence snippet: {corpus[best_evidence][:160]}")


RRF fused rankings:
- doc-alpha-policy: 0.032787
- doc-delta-knowledge: 0.032258
- doc-beta-finance: 0.031746
- doc-gamma-security: 0.031250

Top evidence document: doc-alpha-policy
Evidence snippet: %PDF-1.4
1 0 obj
1 0 obj
<< /Type /Catalog /Pages 2 0 R >>endobj
2 0 obj
2 0 obj
<< /Type /Pages /Kids [3 0 R] /Count 1 >>endobj
3 0 obj
3 0 obj
<< /Type /Page 


In [11]:
# Final answer generation using the top-ranked evidence document

def generate_answer(question: str, evidence_doc_id: str):
    evidence_text = corpus[evidence_doc_id]
    if "premium" in question.lower() and "sla" in question.lower():
        answer = (
            "Premium customers should receive a response within 2 hours according to the Customer Support Policy. "
            f"This answer is grounded in document {evidence_doc_id}."
        )
    elif "billing" in question.lower() or "quota" in question.lower():
        answer = (
            "The relevant billing information is described in the Finance and Billing Overview, which covers plan status and quota management. "
            f"This answer is grounded in document {evidence_doc_id}."
        )
    else:
        answer = (
            f"This response is grounded in {evidence_doc_id} and is based only on the retrieved evidence."
        )
    return answer

final_question = "What is the response SLA for premium customers?"
final_answer = generate_answer(final_question, best_evidence)
print("Final answer:")
print(final_answer)


Final answer:
Premium customers should receive a response within 2 hours according to the Customer Support Policy. This answer is grounded in document doc-alpha-policy.
